In [1]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from astropy.time import Time
import numpy as np
from astropy.visualization import time_support
import matplotlib.dates as mdates

In [2]:
def get_data(dir_files, file_time):

    resources = pd.read_csv(f"{dir_files}/resources_{file_time}.csv")
    db_table  = pd.read_csv(f"{dir_files}/table_size_{file_time}.csv")

    data_dict = {}

    data_dict['t']       = Time(resources['timestamp'].tolist())
    data_dict['t_db']    = Time(db_table['timestamp'].tolist())
    
    data_dict['cpu']     = np.array(resources['cpu_percent'])
    data_dict['mem']     = np.array(resources['memory_mb'])
    data_dict['db_size'] = np.array(db_table['total_size_bytes'])/1e6 # MB
    

    return data_dict

In [3]:
def get_Nsrc(dir_files, file_time):

    file_path = f'{dir_files}/log_{file_time}.txt'

    with open(file_path, "r", encoding="utf-8") as file:
        for line_num, line in enumerate(file, 1):
            if 'Starting code for' in line:
                Nsrcs = int(line.strip().split(' ')[5])

    return Nsrcs

In [4]:
def get_cpu_duration(data, Ncycle):

    idx_high = np.where(data['cpu'] > 10)[0]

    dt_sec = (data['t'][1].mjd - data['t'][0].mjd)*24*3600

    t_high = len(idx_high)*dt_sec/Ncycle

    #print(Ncycle, t_high)

    return t_high

In [5]:
def interp_data(data, tmin, tmax, time_increment_sec=1):
    """Interpolate data onto a uniform time grid.
    
    Parameters
    ----------
    data : dict
        Data dictionary from get_data() containing 't', 'mem', 'cpu', 'db_size'
    tmin : str
        Start time as '2026-09-08 13:45:17.000'
    tmax : str
        End time as '2026-09-08 13:45:17.000'
    time_increment_sec : float
        Time increment in seconds for the uniform grid. Default 5 seconds.
    
    Returns
    -------
    """

    t_min = Time(tmin)
    t_max = Time(tmax)
    
    mjd_min = t_min.mjd
    mjd_max = t_max.mjd
    mjd_grid = np.arange(mjd_min, mjd_max, time_increment_sec / (24*3600))
    t_interp = Time(mjd_grid, format='mjd')
    
    t_original_mjd    = data['t'].mjd
    t_original_mjd_db = data['t_db'].mjd
    
    mem_interp = np.interp(mjd_grid, t_original_mjd, data['mem'])
    cpu_interp = np.interp(mjd_grid, t_original_mjd, data['cpu'])
    db_size_interp = np.interp(mjd_grid, t_original_mjd_db, data['db_size'])
    
    return t_interp, mem_interp, cpu_interp, db_size_interp


In [8]:
def make_plot(dir_base, Nmin, Nmax):
    width  = 30 # Figure width for full-page-width works well with fontsize=26
    height = 20 # Adjust as needed
    fs     = 22 # Fontsize that works well with above figure widths

    time_support()

    fig = plt.figure(figsize=(width,height))
    plt.subplots_adjust(left=0.07, 
                        bottom=0.05, 
                        right=0.99, 
                        top=0.95, 
                        hspace=0.15,
                        wspace=0.2)

    gs = gridspec.GridSpec(ncols=3, nrows=6, figure=fig)
    ax1 = fig.add_subplot(gs[0,0:2])
    ax2 = fig.add_subplot(gs[1,0:2])
    ax3 = fig.add_subplot(gs[2,0:2])
    ax4 = fig.add_subplot(gs[3,0:2])
    ax5 = fig.add_subplot(gs[4,0:2])
    ax6 = fig.add_subplot(gs[5,0:2])
    ax7 = fig.add_subplot(gs[0:2,2])
    ax8 = fig.add_subplot(gs[2:4,2])
    ax9 = fig.add_subplot(gs[4:6,2])
    axs = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9]

    instances = !{'ls '+dir_base}

    Nsrcs    = np.empty(len(instances))
    med_mem  = np.empty(len(instances))
    diff_mem = np.empty(len(instances))
    cpu_high = np.empty(len(instances))

    for i in range(0,len(instances)):

        run_dir = f'{dir_base}/{instances[i]}'
        data = get_data(run_dir, instances[i])

        Nsrcs[i]    = get_Nsrc(run_dir, instances[i])
        med_mem[i]  = np.nanmedian(data['mem'])
        diff_mem[i] = np.nanmax(data['mem']) - med_mem[i]
        cpu_high[i] = get_cpu_duration(data, len(data['t_db']))

        if i == 0:
            tmin = data['t'][0]
            tmax = data['t'][-1]
            t_interp, mem_interp_tot, cpu_interp_tot, db_size_interp_tot = interp_data(data, tmin, tmax)

        t_interp, mem_interp, cpu_interp, db_size_interp = interp_data(data, tmin, tmax)

        mem_interp_tot     = mem_interp_tot + mem_interp
        cpu_interp_tot     = cpu_interp_tot + cpu_interp
        db_size_interp_tot = db_size_interp_tot + db_size_interp

        ax1.plot(data['t'], data['mem'])
        ax3.plot(data['t'], data['cpu'])
        ax5.scatter(data['t_db'], data['db_size'])

        ax1.set_ylabel('Memory (MB)',fontsize=fs)
        ax3.set_ylabel('CPU (%)',fontsize=fs)
        ax5.set_ylabel('Database \n Storage (MB)',fontsize=fs)

    ax2.plot(t_interp,  mem_interp_tot,color='blue', linewidth=2)
    ax4.plot(t_interp,  cpu_interp_tot,color='red', linewidth=2)
    ax6.plot(t_interp,  db_size_interp_tot,color='k', linewidth=2)

    ax2.set_ylabel('Total Memory (MB)',fontsize=fs)
    ax4.set_ylabel('Total CPU (%)',fontsize=fs)
    ax6.set_ylabel('Total Database \n Storage (MB)',fontsize=fs)

    ax7.scatter(Nsrcs, med_mem, s=100, color='blue')
    ax8.scatter(Nsrcs, diff_mem, s=100, color='blue')
    ax9.scatter(Nsrcs, cpu_high, s=100, color='red')

    ax7.set_ylabel('Median Memory (MB)',fontsize=fs)
    ax8.set_ylabel('Max - Median Memory (MB)',fontsize=fs)
    ax9.set_ylabel('Time with high CPU (s)',fontsize=fs)

    for k in range(0,9):
        if k < 6:
            axs[k].set_xlim(t_interp[0],t_interp[-1])
            axs[k].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            axs[k].set_xlabel(' ')
        for spine in axs[k].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        axs[k].tick_params(axis='both', labelsize=fs, left=True, right=True, 
                        top=True, which='both', width=2, length=6)
    ax6.set_xlabel('Time', fontsize=fs)
    ax9.set_xlabel('Number of targets in user database', fontsize=fs)

    title = f'Simulation with {len(instances)} users requesting {Nmin} to {Nmax} targets each'
    fig.suptitle(title, fontsize=fs+10)

    plt.savefig('/home/aordog/Work/candiapl/rubin-sunrise/logs/data_logs/test_benchmark1.pdf')
    plt.close()

    return




In [9]:
dir_base = '/home/aordog/Work/candiapl/rubin-sunrise/logs/data_logs/05_users_N1000_2000/'
make_plot(dir_base, 1000, 2000)
